# 10.2 풀링과 전형적인 CNN 구조 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter10_2_pooling_resnet.ipynb)

책 본문: [10.2 풀링과 전형적인 CNN 구조](https://smhanlab.com/book-ml/kor/ml1/chapter10/2.html)

이 노트북은 책 10.2절의 핵심 주장들을 **모두 코드로 검증**합니다:

1. 4×4 특징지도를 1픽셀 이동해도 `max_pool` 출력이 같은지(이동 둔감성, 본문 수동 계산 `[[1,3],[1,4]]` 검증)
2. 수용장 누적 공식 \(R_l = R_{l-1} + (k_l-1)\prod_{j<l} s_j\)이 본문의 표(5, 6, 14)를 재현하는지, 풀링이 수용장 확장을 어떻게 스케일업하는지
3. LeNet-5 표의 각 행 파라미터 수(156, 2,416, 48,120, 10,164, 850 → 총 61,706)와 `SmallCNN`(1,762)을 10.1절 공식으로 재계산
4. `SmallCNN`이 세로줄 vs 가로줄을 30 에폭에 train/val 100%로 구분하는지
5. **깊이 열화**: 스킵 연결 없는/있는 24층 CNN이 같은 데이터에서 각각 50% / 100%로 수렴하는지
6. 20층 FC에서 스킵 연결만 달랐을 때 \(\|dz/dx\|\)가 2.5e-14 vs 8.2인 "+1 항" 검증

## 1. 4×4 특징지도에서 최대풀링 손으로 계산하기

본문의 예: 1픽셀 오른쪽 이동 전후, \(2\times2\) 최대풀링 결과는 모두 \(\begin{bmatrix}1&3\\1&4\end{bmatrix}\).

In [1]:
def max_pool(feature_map, pool_size=2):
    h, w = len(feature_map), len(feature_map[0])
    out_h, out_w = h // pool_size, w // pool_size
    output = [[0.0] * out_w for _ in range(out_h)]
    for i in range(out_h):
        for j in range(out_w):
            block = [feature_map[i*pool_size + di][j*pool_size + dj]
                     for di in range(pool_size) for dj in range(pool_size)]
            output[i][j] = max(block)
    return output

fm = [[1, 0, 2, 1],
      [1, 0, 3, 1],
      [1, 0, 4, 1],
      [1, 0, 2, 1]]
# 1픽셀 오른쪽 이동: 0번째 열이 1로 채워지고, 나머지 열이 1칸 오른쪽으로
fm_shifted = [[1, 1, 0, 2],
              [1, 1, 0, 3],
              [1, 1, 0, 4],
              [1, 1, 0, 2]]

p1, p2 = max_pool(fm), max_pool(fm_shifted)
print("원본 max_pool:    ", p1)
print("1픽셀 이동 max_pool:", p2)
assert p1 == p2 == [[1, 3], [1, 4]], "본문의 수동 계산과 다름"
print("\n두 결과가 같고 [[1,3],[1,4]] -> 본문의 수동 계산과 일치")

# 풀링 없이(그대로)라면? 16개 숫자 중 8개가 다름
diff = sum(a != b for row_a, row_b in zip(fm, fm_shifted) for a, b in zip(row_a, row_b))
print(f"풀링 없이 비교하면 16개 원소 중 {diff}개가 다름 (1픽셀 이동이 출력 전체를 바꿈)")

원본 max_pool:     [[1, 3], [1, 4]]
1픽셀 이동 max_pool: [[1, 3], [1, 4]]

두 결과가 같고 [[1,3],[1,4]] -> 본문의 수동 계산과 일치
풀링 없이 비교하면 16개 원소 중 12개가 다름 (1픽셀 이동이 출력 전체를 바꿈)


## 2. 수용장 누적 공식 검증

\[R_l = R_{l-1} + (k_l - 1) \cdot \prod_{j < l} s_j, \qquad R_0 = 1\]

In [2]:
def receptive_field(layers):
    """layers: [(name, k, s), ...] -> 각 층의 누적 수용장 크기 리스트"""
    rf, stride_prod = 1, 1
    out = []
    for name, k, s in layers:
        rf = rf + (k - 1) * stride_prod
        out.append((name, rf))
        stride_prod *= s
    return out

# 본문의 예 (입력 32x32): Conv5 -> Pool2(s=2) -> Conv5
rf = receptive_field([("Conv 5x5 s=1", 5, 1),
                      ("MaxPool 2x2 s=2", 2, 2),
                      ("Conv 5x5 s=1", 5, 1)])
for name, r in rf:
    print(f"  {name:16s} -> R = {r}")
assert [r for _, r in rf] == [5, 6, 14], "본문의 표(5, 6, 14)와 다름"
print("\n본문의 표(5 -> 6 -> 14)와 정확히 일치")

# 3x3 s=1 만을 l개 쌓으면 R = 1 + 2l
def stack3x3(l):
    return receptive_field([("Conv 3x3 s=1", 3, 1)] * l)[-1][1]
assert stack3x3(10) == 21 and stack3x3(20) == 41
print("3x3 s=1만: 10층 -> R=21, 20층 -> R=41 (선형: 1+2l)")

# 풀링을 끼우면 그 뒤 확장이 스케일업: Conv5,Pool2,Conv5 x 반복 (LeNet 계열)
def lenet_style(depth_blocks):
    layers = []
    for _ in range(depth_blocks):
        layers.append(("Conv 5x5 s=1", 5, 1))
        layers.append(("MaxPool 2x2 s=2", 2, 2))
    return receptive_field(layers)[-1][1]
print("LeNet 계열 [Conv5,Pool2] 블록:", {b: lenet_style(b) for b in (1, 2, 3, 4)})

  Conv 5x5 s=1     -> R = 5
  MaxPool 2x2 s=2  -> R = 6
  Conv 5x5 s=1     -> R = 14

본문의 표(5 -> 6 -> 14)와 정확히 일치
3x3 s=1만: 10층 -> R=21, 20층 -> R=41 (선형: 1+2l)
LeNet 계열 [Conv5,Pool2] 블록: {1: 6, 2: 16, 3: 36, 4: 76}


## 3. 그림 1: 풀링이 수용장 확장을 어떻게 가속하는가

같은 깊이의 네트워크에서, 풀링 없이(스트라이드 1 합성곱만) 쌓을 때와 풀링을 주기적으로 끼울 때 수용장 크기를 비교합니다. 풀링(스트라이드 2) 한 번이 *이후 모든* 확장을 2배 스케일업시킵니다.

In [3]:
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)

def rf_curve(layers):
    rf, sp = 1, 1
    out = []
    for name, k, s in layers:
        rf = rf + (k - 1) * sp
        out.append(rf)
        sp *= s
    return out

plain_layers  = [("Conv 3x3 s=1", 3, 1)] * 16
pooled_layers = []
for i in range(4):            # [Conv3,Conv3,Pool2] x 4 = 같은 16개 연산
    pooled_layers += [("Conv 3x3 s=1", 3, 1), ("Conv 3x3 s=1", 3, 1), ("MaxPool 2x2 s=2", 2, 2)]

rf_plain  = rf_curve(plain_layers)
rf_pooled = rf_curve(pooled_layers)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(rf_plain) + 1), rf_plain, "-o", ms=4, label="16 layers of 3x3 s=1 (no pooling)")
ax.plot(range(1, len(rf_pooled) + 1), rf_pooled, "-s", ms=4, label="[Conv3, Conv3, Pool2] x 4")
ax.axhline(32, ls=":", c="gray", label="input image size (32px)")
ax.set_xlabel("Layer (cumulative conv+pooling)")
ax.set_ylabel("Receptive field size (original image pixels)")
ax.set_title("Cumulative receptive field: pooling (stride 2) doubles downstream expansion")
ax.legend()
ax.set_ylim(0, 80)
fig.tight_layout()
fig.savefig(f"{IMG}/ch10_2_receptive_field.svg")
print("저장:", f"{IMG}/ch10_2_receptive_field.svg")
print("16층 끝: 풀링 없음 R =", rf_plain[-1], ", 풀링 있음 R =", rf_pooled[-1])

저장: /home/smhan/book-ml/kor/src/images/ch10_2_receptive_field.svg
16층 끝: 풀링 없음 R = 33 , 풀링 있음 R = 76


## 4. 파라미터 수 재계산: LeNet-5 표 + SmallCNN

10.1절 공식: 합성곱 \((k^2 C_{in} + 1) C_{out}\), 완전연결 \(C_{in} C_{out} + C_{out}\).

In [4]:
import torch
import torch.nn as nn

# ---- LeNet-5 (본문 표) ----
# 합성곱: (k^2 * C_in + 1) * C_in_out   완전연결: C_in * C_out + C_out
lenet5 = [
    # (층, 설명, C_in, C_out, 종류)
    ("C1",  "5x5 Conv + sigmoid",   1,  6,   "conv"),
    ("C3",  "5x5 Conv + sigmoid",   6,  16,  "conv"),
    ("C5",  "5x5 Conv + sigmoid",   16, 120, "conv"),
    ("F6",  "FC(tanh)",             120, 84, "fc"),   # C5 출력(1x1x120)을 펼침
    ("OUT", "FC(softmax)",          84,  10, "fc"),
]
total = 0
print("LeNet-5 파라미터 수 (10.1절 공식 대입):")
for name, op, cin, cout, kind in lenet5:
    if kind == "conv":
        n = (5 * 5 * cin + 1) * cout
    else:
        n = cin * cout + cout
    total += n
    print(f"  {name:4s} {op:18s} -> {n:>7,}")
print(f"  합계: {total:,}")
assert total == 156 + 2416 + 48120 + 10164 + 850 == 61706, "본문 표 합계(61,706)와 다름"
fc_total = 10164 + 850
print(f"  완전연결(F6+OUT) 비중: {fc_total/total:.0%}")

# ---- SmallCNN (본문 표) ----
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(16*4*4, 2)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

m = SmallCNN()
n_conv1 = m.conv1.weight.numel() + m.conv1.bias.numel()
n_conv2 = m.conv2.weight.numel() + m.conv2.bias.numel()
n_fc    = m.fc.weight.numel() + m.fc.bias.numel()
n_total = n_conv1 + n_conv2 + n_fc
print(f"\nSmallCNN: Conv1={n_conv1}, Conv2={n_conv2}, FC={n_fc}, 총={n_total}")
assert (n_conv1, n_conv2, n_fc, n_total) == (80, 1168, 514, 1762), "본문 표(80, 1168, 514, 1762)와 다름"
print("본문 표(80 + 1,168 + 514 = 1,762)와 정확히 일치")

LeNet-5 파라미터 수 (10.1절 공식 대입):
  C1   5x5 Conv + sigmoid ->     156
  C3   5x5 Conv + sigmoid ->   2,416
  C5   5x5 Conv + sigmoid ->  48,120
  F6   FC(tanh)           ->  10,164
  OUT  FC(softmax)        ->     850
  합계: 61,706
  완전연결(F6+OUT) 비중: 18%

SmallCNN: Conv1=80, Conv2=1168, FC=514, 총=1762
본문 표(80 + 1,168 + 514 = 1,762)와 정확히 일치


## 5. SmallCNN 학습: 세로줄 vs 가로줄

16×16, 잡음 섞인 세로줄/가로줄 이미지. 학습 전 정확도는 동전 던지기 수준이어야 하고, 30 에폭 후 train/val 100%에 도달해야 합니다.

In [5]:
import numpy as np
import torch
import torch.nn as nn

def make_line_data(n, size=16, noise=0.5, seed=0):
    """i%2==0: 세로줄, i%2==1: 가로줄. 줄 위치는 무작위(3~size-3)."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for i in range(n):
        img = rng.normal(0, noise, (size, size))
        pos = int(rng.integers(3, size - 2))
        if i % 2 == 0:
            img[:, pos] += 4.0      # 세로줄
        else:
            img[pos, :] += 4.0      # 가로줄
        X.append(img[None] / 10.0)
        y.append(i % 2)
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y))

Xtr, ytr = make_line_data(600, seed=0)
Xva, yva = make_line_data(200, seed=1)
print("train:", tuple(Xtr.shape), " val:", tuple(Xva.shape))

torch.manual_seed(0)
model = SmallCNN()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
lossfn = nn.CrossEntropyLoss()

with torch.no_grad():
    acc0 = (model(Xva).argmax(1) == yva).float().mean().item()
print(f"학습 전 val 정확도: {acc0:.3f} (동전 던지기 수준)")

for ep in range(30):
    model.train()
    opt.zero_grad()
    loss = lossfn(model(Xtr), ytr)
    loss.backward()
    opt.step()
    if ep in (0, 9, 19, 29):
        model.eval()
        with torch.no_grad():
            atr = (model(Xtr).argmax(1) == ytr).float().mean().item()
            ava = (model(Xva).argmax(1) == yva).float().mean().item()
        print(f"  ep {ep:2d}: loss={loss.item():.4f}  train={atr:.3f}  val={ava:.3f}")

model.eval()
with torch.no_grad():
    atr = (model(Xtr).argmax(1) == ytr).float().mean().item()
    ava = (model(Xva).argmax(1) == yva).float().mean().item()
print(f"최종: train={atr:.3f}  val={ava:.3f}")
assert acc0 < 0.6 and atr == 1.0 and ava == 1.0, "본문(52% -> 100%/100%)과 다름"
print("본문과 일치: 학습 전 ~50% -> 30 에폭 후 train/val 100%")

train: (600, 1, 16, 16)  val: (200, 1, 16, 16)


학습 전 val 정확도: 0.500 (동전 던지기 수준)
  ep  0: loss=0.6931  train=0.500  val=0.500
  ep  9: loss=0.4080  train=1.000  val=1.000
  ep 19: loss=0.0411  train=1.000  val=1.000
  ep 29: loss=0.0012  train=1.000  val=1.000
최종: train=1.000  val=1.000
본문과 일치: 학습 전 ~50% -> 30 에폭 후 train/val 100%


## 6. 깊이 열화: 스킵 연결 없는/있는 24층 CNN

3×3 Conv-Conv 블록 12개를 쌓은(=24개 합성곱 층) 두 네트워크를 *같은* 데이터로 학습. 스킵 연결이 없으면 loss가 \(\ln 2 \approx 0.693\)(동전 던지기)에 고정되고, 있으면 100%에 도달해야 합니다.

In [6]:
import torch.nn as nn
import torch

class ResBlock(nn.Module):
    """y = F(x) + x  (F: Conv-ReLU-Conv)"""
    def __init__(self, ch):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(),
                               nn.Conv2d(ch, ch, 3, padding=1))
    def forward(self, x):
        return torch.relu(self.b(x) + x)

def make_net(residual, blocks=12, ch=8, seed=0):
    torch.manual_seed(seed)
    layers = [nn.Conv2d(1, ch, 3, padding=1), nn.ReLU()]
    for _ in range(blocks):
        if residual:
            layers.append(ResBlock(ch))
        else:
            layers.append(nn.Sequential(nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(),
                                        nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU()))
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(ch, 2)]
    return nn.Sequential(*layers)

def train_curve(residual, epochs=25, seed=0):
    model = make_net(residual, blocks=12, ch=8, seed=seed)
    opt = torch.optim.Adam(model.parameters(), lr=0.005)
    lossfn = nn.CrossEntropyLoss()
    hist = []
    model.train()
    for ep in range(epochs):
        opt.zero_grad()
        loss = lossfn(model(Xtr), ytr)
        loss.backward()
        opt.step()
        with torch.no_grad():
            acc = (model(Xtr).argmax(1) == ytr).float().mean().item()
        hist.append((loss.item(), acc))
    return hist

h_plain = train_curve(residual=False)
h_res   = train_curve(residual=True)

for tag, h in [("평범(스킵 없음)", h_plain), ("잔차(스킵 있음)", h_res)]:
    print(f"{tag}:")
    for i in (0, 5, 10, 15, 20, 24):
        print(f"   ep {i:2d}: loss={h[i][0]:.4f}  train_acc={h[i][1]:.3f}")
    print()

plain_final, res_final = h_plain[-1], h_res[-1]
print(f"최종: 평범 loss={plain_final[0]:.4f}(~ln2={np.log(2):.4f}, acc={plain_final[1]:.3f})  "
      f"vs  잔차 loss={res_final[0]:.4f}(acc={res_final[1]:.3f})")
assert plain_final[1] < 0.6 and res_final[1] == 1.0, "본문(50% 고정 vs 100%)과 다름"
print("본문과 일치: 스킵 없음은 동전 던지기 수준에 고정, 스킵 있음은 100% 도달")

평범(스킵 없음):
   ep  0: loss=0.6974  train_acc=0.500
   ep  5: loss=0.6935  train_acc=0.500
   ep 10: loss=0.6931  train_acc=0.500
   ep 15: loss=0.6931  train_acc=0.500
   ep 20: loss=0.6932  train_acc=0.500
   ep 24: loss=0.6932  train_acc=0.500

잔차(스킵 있음):
   ep  0: loss=0.7073  train_acc=0.500
   ep  5: loss=0.6900  train_acc=1.000
   ep 10: loss=0.6544  train_acc=0.500
   ep 15: loss=0.3552  train_acc=1.000
   ep 20: loss=0.2752  train_acc=1.000
   ep 24: loss=0.2439  train_acc=1.000

최종: 평범 loss=0.6932(~ln2=0.6931, acc=0.500)  vs  잔차 loss=0.2439(acc=1.000)
본문과 일치: 스킵 없음은 동전 던지기 수준에 고정, 스킵 있음은 100% 도달


## 7. 그림 2: 깊이 열화 학습 곡선

같은 24층 구조, 같은 데이터 — 스킵 연결 여부만으로 학습 곡선이 갈립니다.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
eps = list(range(25))

axes[0].plot(eps, [h[0] for h in h_plain], "-o", ms=3, label="No skip (24 layers)")
axes[0].plot(eps, [h[0] for h in h_res],   "-s", ms=3, label="With skip (24 layers)")
axes[0].axhline(np.log(2), ls=":", c="gray")
axes[0].text(1, np.log(2) + 0.004, "ln 2 = coin flip", fontsize=9, c="gray")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("train cross-entropy loss")
axes[0].set_title("Loss: without skips, stuck at ln 2")
axes[0].legend()

axes[1].plot(eps, [h[1] for h in h_plain], "-o", ms=3, label="No skip (24 layers)")
axes[1].plot(eps, [h[1] for h in h_res],   "-s", ms=3, label="With skip (24 layers)")
axes[1].axhline(0.5, ls=":", c="gray")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("train accuracy")
axes[1].set_title("Accuracy: skip connections take 50% -> 100%")
axes[1].legend()

fig.suptitle("Depth degradation: same architecture and data, only skip connections differed")
fig.tight_layout()
fig.savefig(f"{IMG}/ch10_2_depth_curves.svg")
print("저장:", f"{IMG}/ch10_2_depth_curves.svg")

저장: /home/smhan/book-ml/kor/src/images/ch10_2_depth_curves.svg


## 8. "+1" 항: 20층 FC의 그래디언트

\(y=F(x)+x\) \(\Rightarrow \partial y/\partial x = \partial F/\partial x + 1\). L개의 잔차 블록을 쌓으면 \(\partial x_L/\partial x_0 = \prod_l (\partial F_l/\partial x_{l-1} + 1)\)로, 전개하면 정확히 **1**인 항이 하나 남는다. 같은 가중치(표준편차 0.1)의 20층 FC에서 출력에 대한 입력 그래디언트 노름을 잰다.

In [8]:
torch.manual_seed(3)
H, L = 10, 20
Ws = [torch.randn(H, H) * 0.1 for _ in range(L)]
x0 = torch.tensor([[1.0, 2.0, -1.0, 0.5, 3.0, -2.0, 1.5, 0.0, -0.5, 2.5]])

x = x0.clone().requires_grad_(True)
z = x
for W in Ws:
    z = torch.relu(W @ z.T).T
z.sum().backward()
g_plain = x.grad.norm().item()

x = x0.clone().requires_grad_(True)
z = x
for W in Ws:
    z = z + 0.5 * torch.relu(W @ z.T).T
z.sum().backward()
g_res = x.grad.norm().item()

print(f"스킵 없는 20층: |dz/dx| = {g_plain:.3e}")
print(f"스킵 있는 20층: |dz/dx| = {g_res:.3e}")
print(f"비: {g_res / g_plain:.2e} 배")
assert g_plain < 1e-8, "스킵 없음이 소실되어야 함"
assert g_res > 1.0, "스킵 있음이 신호를 유지해야 함"
print("본문(2.5e-14 vs 8.2, ~330조 배)과 일치: 스킵 연결 없이는 그래디언트가 사실상 0")

스킵 없는 20층: |dz/dx| = 2.491e-14
스킵 있는 20층: |dz/dx| = 8.198e+00
비: 3.29e+14 배
본문(2.5e-14 vs 8.2, ~330조 배)과 일치: 스킵 연결 없이는 그래디언트가 사실상 0


## 요약

| 검증 | 본문 주장 | 결과 |
|---|---|---|
| 1 | 1픽셀 이동 전후 maxpool == [[1,3],[1,4]] | O |
| 2 | 수용장 5 -> 6 -> 14, 풀링이 확장 스케일업 | O |
| 3 | LeNet-5 61,706 / SmallCNN 1,762 | O |
| 4 | SmallCNN 30에폭 train/val 100% | O |
| 5 | 24층: 스킵 없음 50% 고정 / 스킵 있음 100% | O |
| 6 | 20층 FC 그래디언트 2.5e-14 vs 8.2 | O |